# Cloud Height Binary Segmentation Baseline

This notebook implements a binary segmentation model to detect cloud presence in satellite imagery.
- **Task**: Binary classification (cloud vs. no-cloud)
- **Model**: Custom U-Net with binary output
- **Metrics**: Binary Accuracy, F1-Score, IoU (Jaccard Index)

## 1. Setup and Imports

In [ ]:
#!pip install pytorch-lightning segmentation_models_pytorch torchmetrics

import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl

import torchvision.models as models
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import segmentation_models_pytorch as smp
import tqdm
import pandas as pd

from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    RichProgressBar,
    TQDMProgressBar
)
from torchmetrics.classification import (
    BinaryAccuracy,
    BinaryF1Score,
    BinaryJaccardIndex,
)

print("✅ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch Lightning version: {pl.__version__}")

## 2. Configuration

In [ ]:
# Data and paths
DATA_DIRECTORY = "/explore/nobackup/projects/pix4dcloud/szhang16/abiChips/GOES-16"
WORK_DIR = "/explore/nobackup/projects/ilab/projects/SatVisionPix4D/development/cloud_height_baseline"

# Training hyperparameters
BATCH_SIZE = 256
NUM_WORKERS = 40
MAX_EPOCHS = 6000
LEARNING_RATE = 1e-4

# Model architecture
TARGET_HEIGHT = 96
TARGET_WIDTH = 40
IN_CHANNELS = 16  # Number of ABI channels
NUM_CLASSES = 1   # Binary segmentation (single output channel)

# Visualization
NUM_SAMPLES_TO_TEST = 5

# Set random seed for reproducibility
pl.seed_everything(42)

print("\n Configuration:")
print(f"  Data directory: {DATA_DIRECTORY}")
print(f"  Work directory: {WORK_DIR}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Target size: ({TARGET_HEIGHT}, {TARGET_WIDTH})")
print(f"  Task: Binary Cloud Segmentation")

## 3. Dataset Definition

In [ ]:
class CloudSatDataset(Dataset):
    """Dataset for CloudSat binary cloud mask segmentation."""
    
    def __init__(self, file_paths, target_size=(96, 40)):
        self.file_paths = file_paths
        self.target_height, self.target_width = target_size

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        try:
            with np.load(self.file_paths[idx], allow_pickle=True) as data:
                # Load and normalize ABI chip
                abi_chip = data['chip'].astype(np.float32)

                # Normalize each channel independently
                for i in range(abi_chip.shape[2]):
                    channel = abi_chip[:, :, i]
                    min_val = channel.min()
                    max_val = channel.max()
                    if max_val > min_val:
                        abi_chip[:, :, i] = (channel - min_val) / (max_val - min_val)
                
                # Transpose to channel-first format: (H, W, C) -> (C, H, W)
                abi_chip = np.transpose(abi_chip, (2, 0, 1))

                # Load binary cloud mask
                cloud_mask_raw = data['data'].item()['Cloud_mask_binary'].astype(np.float32)
                cloud_mask_raw = cloud_mask_raw[:, :34]  # Crop to width 34
                
                # Pad mask to target size
                pad_height_total = self.target_height - cloud_mask_raw.shape[0]
                pad_top = pad_height_total // 2
                pad_bottom = pad_height_total - pad_top
                pad_width_total = self.target_width - cloud_mask_raw.shape[1]
                pad_left = pad_width_total // 2
                pad_right = pad_width_total - pad_left
                
                cloud_mask_padded = np.pad(
                    cloud_mask_raw,
                    ((pad_top, pad_bottom), (pad_left, pad_right)),
                    'constant',
                    constant_values=0
                )
               
                return {
                    "chip": torch.from_numpy(abi_chip),
                    "mask": torch.from_numpy(cloud_mask_padded),
                    "path": self.file_paths[idx]
                }
        except Exception as e:
            print(f"⚠️  Error loading {self.file_paths[idx]}: {e}")
            return None

## 4. Data Module

In [ ]:
class CloudSatDataModule(pl.LightningDataModule):
    """PyTorch Lightning DataModule for CloudSat dataset."""
    
    def __init__(self, data_dir, batch_size=16, num_workers=0, train_val_test_split=(0.8, 0.1, 0.1)):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_val_test_split = train_val_test_split
        self.file_paths = sorted(glob.glob(os.path.join(self.data_dir, '*.npz')))
        
        print(f"\n📂 Found {len(self.file_paths)} data files in {data_dir}")

    def setup(self, stage=None):
        n_total = len(self.file_paths)
        n_train = int(n_total * self.train_val_test_split[0])
        n_val = int(n_total * self.train_val_test_split[1])
       
        self.train_files = self.file_paths[:n_train]
        self.val_files = self.file_paths[n_train : n_train + n_val]
        self.test_files = self.file_paths[n_train + n_val :]

        self.train_dataset = CloudSatDataset(self.train_files)
        self.val_dataset = CloudSatDataset(self.val_files)
        self.test_dataset = CloudSatDataset(self.test_files)
        
        print(f"\n📊 Data split:")
        print(f"  Train: {len(self.train_files)} files")
        print(f"  Val:   {len(self.val_files)} files")
        print(f"  Test:  {len(self.test_files)} files")

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset, batch_size=self.batch_size,
            shuffle=True, num_workers=self.num_workers,
            pin_memory=True, collate_fn=self._collate_fn
        )
    
    def val_dataloader(self):
        return DataLoader(
            self.val_dataset, batch_size=self.batch_size,
            num_workers=self.num_workers, pin_memory=True,
            collate_fn=self._collate_fn
        )
    
    def test_dataloader(self):
        return DataLoader(
            self.test_dataset, batch_size=self.batch_size,
            num_workers=self.num_workers, pin_memory=True,
            collate_fn=self._collate_fn
        )

    def _collate_fn(self, batch):
        """Filter out None samples and collate."""
        batch = list(filter(lambda x: x is not None, batch))
        if not batch:
            return None
        return torch.utils.data.dataloader.default_collate(batch)

## 5. Loss Functions

In [ ]:
class DiceLoss(nn.Module):
    """Dice Loss for binary segmentation."""
    
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        intersection = torch.sum(probs * targets)
        cardinality = torch.sum(probs + targets)
        dice_score = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return 1. - dice_score

## 6. Model Architecture

In [ ]:
class CustomUNET(nn.Module):
    """Custom U-Net architecture for binary segmentation."""
    
    def __init__(self, in_channels=16, decoder_classes=64, num_classes=1, head_dropout=0.2, output_shape=(96, 40)):
        super().__init__()
        self.layers = [in_channels, 64, 128, 256, 512, 1024]

        # Encoder (downsampling)
        self.double_conv_downs = nn.ModuleList(
            [self.__double_conv(layer, layer_n) for layer, layer_n in zip(self.layers[:-1], self.layers[1:])])

        # Decoder (upsampling)
        self.up_trans = nn.ModuleList(
            [nn.ConvTranspose2d(layer, layer_n, kernel_size=2, stride=2)
             for layer, layer_n in zip(self.layers[::-1][:-2], self.layers[::-1][1:-1])])

        self.double_conv_ups = nn.ModuleList(
            [self.__double_conv(layer, layer//2) for layer in self.layers[::-1][:-2]])

        self.max_pool_2x2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Final layers
        self.final_conv = nn.Conv2d(64, decoder_classes, kernel_size=1)

        self.prediction_head = nn.Sequential(
            nn.Conv2d(decoder_classes, num_classes, kernel_size=3, stride=1, padding=1),
            nn.Dropout(head_dropout),
            nn.Upsample(size=output_shape, mode='bilinear', align_corners=False)
        )

    def __double_conv(self, in_channels, out_channels):
        """Double convolution block."""
        conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        return conv

    def forward(self, x):
        # Encoder path
        concat_layers = []

        for down in self.double_conv_downs:
            x = down(x)
            if down != self.double_conv_downs[-1]:
                concat_layers.append(x)
                x = self.max_pool_2x2(x)

        concat_layers = concat_layers[::-1]

        # Decoder path
        for up_trans, double_conv_up, concat_layer in zip(self.up_trans, self.double_conv_ups, concat_layers):
            x = up_trans(x)
            if x.shape != concat_layer.shape:
                x = TF.resize(x, concat_layer.shape[2:])

            concatenated = torch.cat((concat_layer, x), dim=1)
            x = double_conv_up(concatenated)

        x = self.final_conv(x)
        x = self.prediction_head(x)

        return x

    def predict(self, x):
        """Generate binary predictions."""
        x = self.forward(x)
        x = torch.sigmoid(x)
        x = (x > 0.5).float()
        return x

## 7. Lightning Module

In [ ]:
class CustomUNETLightning(pl.LightningModule):
    """PyTorch Lightning wrapper for binary cloud segmentation."""

    def __init__(
        self,
        in_channels=16,
        num_classes=1,
        target_height=96,
        target_width=40,
        lr=1e-4,
        dice_weight=0.5,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.model = CustomUNET(
            in_channels=self.hparams.in_channels,
            num_classes=self.hparams.num_classes,
            decoder_classes=64,
            head_dropout=0.2,
            output_shape=(self.hparams.target_height, self.hparams.target_width)
        )

        # Binary segmentation losses
        self.ce_loss = nn.BCEWithLogitsLoss()
        self.dice_loss = DiceLoss()
        self.dice_weight = dice_weight

        # Binary metrics
        threshold = 0.5
        self.train_acc = BinaryAccuracy(threshold=threshold)
        self.val_acc   = BinaryAccuracy(threshold=threshold)
        self.test_acc  = BinaryAccuracy(threshold=threshold)

        self.train_f1 = BinaryF1Score(threshold=threshold)
        self.val_f1   = BinaryF1Score(threshold=threshold)
        self.test_f1  = BinaryF1Score(threshold=threshold)

        self.train_iou = BinaryJaccardIndex(threshold=threshold)
        self.val_iou   = BinaryJaccardIndex(threshold=threshold)
        self.test_iou  = BinaryJaccardIndex(threshold=threshold)

    def forward(self, x):
        return self.model(x)

    def _common_step(self, batch, batch_idx, stage: str):
        if batch is None:
            return None

        chips, masks = batch["chip"], batch["mask"]  # masks: [B, H, W] binary
        logits = self.forward(chips)                 # logits: [B, 1, H, W]
        logits = logits.squeeze(1)                   # [B, H, W]

        # Compute losses
        ce = self.ce_loss(logits, masks.float())
        dice = self.dice_loss(logits, masks)
        loss = self.dice_weight * dice + (1 - self.dice_weight) * ce

        # Generate predictions
        preds = torch.sigmoid(logits)
        preds = (preds > 0.5).float()
        
        bs = chips.size(0)

        # Update metrics
        if stage == "train":
            self.train_acc.update(preds, masks)
            self.train_f1.update(preds, masks)
            self.train_iou.update(preds, masks)
        elif stage == "val":
            self.val_acc.update(preds, masks)
            self.val_f1.update(preds, masks)
            self.val_iou.update(preds, masks)
        elif stage == "test":
            self.test_acc.update(preds, masks)
            self.test_f1.update(preds, masks)
            self.test_iou.update(preds, masks)

        # Log losses
        self.log(f"{stage}_loss", loss, prog_bar=True, 
                on_step=True if stage == "train" else False, 
                on_epoch=True, batch_size=bs, 
                sync_dist=(stage != "train"))
        self.log(f"{stage}_ce_loss", ce, prog_bar=False, 
                on_step=False, on_epoch=True, batch_size=bs, 
                sync_dist=(stage != "train"))
        self.log(f"{stage}_dice_loss", dice, prog_bar=False, 
                on_step=False, on_epoch=True, batch_size=bs, 
                sync_dist=(stage != "train"))

        # Log metrics
        if stage == "train":
            self.log("train_acc", self.train_acc, prog_bar=True, 
                    on_step=False, on_epoch=True, batch_size=bs)
            self.log("train_f1",  self.train_f1,  prog_bar=True, 
                    on_step=False, on_epoch=True, batch_size=bs)
            self.log("train_iou", self.train_iou, prog_bar=True, 
                    on_step=False, on_epoch=True, batch_size=bs)
        elif stage == "val":
            self.log("val_acc", self.val_acc, prog_bar=True, 
                    on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)
            self.log("val_f1",  self.val_f1,  prog_bar=True, 
                    on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)
            self.log("val_iou", self.val_iou, prog_bar=True, 
                    on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)
        elif stage == "test":
            self.log("test_acc", self.test_acc, prog_bar=True, 
                    on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)
            self.log("test_f1",  self.test_f1,  prog_bar=True, 
                    on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)
            self.log("test_iou", self.test_iou, prog_bar=True, 
                    on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)

        return loss

    def training_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, "train")

    def validation_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, "val")

    def test_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, "test")

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

## 8. Visualization Functions

In [ ]:
def visualize_prediction(chip_tensor, true_mask_padded, pred_mask_padded, file_path, save_dir):
    """Visualize binary cloud segmentation results."""
    chip = chip_tensor[0].cpu().numpy()
    true_mask = true_mask_padded.cpu().numpy()
    pred_mask = pred_mask_padded.cpu().numpy()
   
    # Unpad to original size
    original_height, original_width = 91, 34
    pad_height_total = true_mask.shape[0] - original_height
    pad_top = pad_height_total // 2
    pad_width_total = true_mask.shape[1] - original_width
    pad_left = pad_width_total // 2
   
    true_mask_unpadded = true_mask[pad_top : pad_top + original_height, 
                                   pad_left : pad_left + original_width]
    pred_mask_unpadded = pred_mask[pad_top : pad_top + original_height, 
                                   pad_left : pad_left + original_width]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 6))
    fig.suptitle(f"File: {os.path.basename(file_path)}", fontsize=16)

    # Input chip
    ax1.imshow(chip, cmap='gray')
    ax1.set_title("Input ABI Chip (Channel 1)", fontsize=14)
    ax1.axis('off')

    # Ground truth
    plot_binary_curtain(ax2, true_mask_unpadded, "Ground Truth (Binary)", fig)
    
    # Prediction
    plot_binary_curtain(ax3, pred_mask_unpadded, "Prediction (Binary)", fig)

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Save figure
    output_filename = os.path.basename(file_path).replace('.npz', '.png')
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, output_filename)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"💾 Saved visualization to: {save_path}")
    

def plot_binary_curtain(ax, mask, title, fig):
    """Plot binary cloud mask as curtain plot."""
    zz = mask.T
   
    x_axis_points = np.arange(zz.shape[1])
    y_axis_km = np.arange(0, zz.shape[0] * 0.5, 0.5)

    # Use binary colormap (white=no cloud, blue=cloud)
    mesh = ax.pcolormesh(x_axis_points, y_axis_km, zz, 
                        cmap='Blues', shading='nearest', 
                        vmin=0, vmax=1)
   
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Point Along Track", fontsize=12)
    ax.set_ylabel("Height (km)", fontsize=12)
    ax.set_ylim(0, zz.shape[0] * 0.5)
    ax.grid(True, linestyle='--', alpha=0.5)
   
    # Binary colorbar
    cbar = fig.colorbar(mesh, ax=ax, ticks=[0, 1])
    cbar.set_ticklabels(['No Cloud', 'Cloud'])


def calculate_binary_iou(pred_mask, true_mask):
    """Calculate IoU for binary segmentation."""
    pred_mask = pred_mask.flatten()
    true_mask = true_mask.flatten()
    
    # Class 0 (No Cloud)
    intersection_0 = np.sum((pred_mask == 0) & (true_mask == 0))
    union_0 = np.sum((pred_mask == 0) | (true_mask == 0))
    iou_0 = intersection_0 / union_0 if union_0 > 0 else 0.0
    
    # Class 1 (Cloud)
    intersection_1 = np.sum((pred_mask == 1) & (true_mask == 1))
    union_1 = np.sum((pred_mask == 1) | (true_mask == 1))
    iou_1 = intersection_1 / union_1 if union_1 > 0 else 0.0
    
    # Mean IoU
    mIoU = (iou_0 + iou_1) / 2
    
    return mIoU, np.array([iou_0, iou_1])

## 9. Training Setup

In [ ]:
# Initialize data module
datamodule = CloudSatDataModule(
    data_dir=DATA_DIRECTORY, 
    batch_size=BATCH_SIZE, 
    num_workers=NUM_WORKERS
)

# Initialize model
model = CustomUNETLightning(
    in_channels=IN_CHANNELS, 
    num_classes=NUM_CLASSES, 
    target_height=TARGET_HEIGHT, 
    target_width=TARGET_WIDTH, 
    lr=LEARNING_RATE, 
    dice_weight=0.5
)

# Callbacks
checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath=os.path.join(WORK_DIR, 'checkpoints'),
    filename='cloudsat-binary-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3,
    mode='min',
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=True,
    mode='min'
)

print("\n✅ Training setup complete")

## 10. Train Model

In [ ]:
trainer = pl.Trainer(
    callbacks=[checkpoint_callback, early_stop_callback, TQDMProgressBar(refresh_rate=20)],
    max_epochs=MAX_EPOCHS,
    accelerator='auto',
    log_every_n_steps=10,
    default_root_dir=WORK_DIR,
    gradient_clip_val=1.0,  # Add gradient clipping
)

print("\n🚀 Starting training...\n")
trainer.fit(model, datamodule)

## 11. Model Evaluation

In [ ]:
# Load best model
best_model_path = checkpoint_callback.best_model_path
print(f"\n📂 Loading best model: {os.path.basename(best_model_path)}")

trained_model = CustomUNETLightning.load_from_checkpoint(best_model_path)
trained_model.eval()

# Get device
device = next(trained_model.parameters()).device
print(f"Model device: {device}")

# Setup test data
datamodule.setup('test')
test_loader = datamodule.test_dataloader()

all_preds = []
all_true_masks = []

print(f"\n🧪 Evaluating on {len(datamodule.test_files)} test samples...\n")

with torch.no_grad():
    for batch in tqdm.tqdm(test_loader, desc="Testing"):
        if batch is None:
            continue
            
        chips = batch['chip'].float().to(device)
        true_masks = batch['mask'].float().to(device)
        
        # Get predictions
        logits = trained_model(chips)
        probs = torch.sigmoid(logits.squeeze(1))
        predicted_masks = (probs > 0.5).float()
        
        all_preds.append(predicted_masks.cpu().numpy())
        all_true_masks.append(true_masks.cpu().numpy())

## 12. Calculate Metrics

In [ ]:
if all_preds:
    all_preds = np.concatenate(all_preds, axis=0)
    all_true_masks = np.concatenate(all_true_masks, axis=0)

    # Calculate binary IoU
    mean_iou, class_iou = calculate_binary_iou(all_preds, all_true_masks)
    
    # Create results table
    CLASS_NAMES = ['No Cloud', 'Cloud']
    iou_data = {'Class': CLASS_NAMES, 'IoU': class_iou}
    iou_df = pd.DataFrame(iou_data)
    
    # Print results
    print("\n" + "="*60)
    print("📊 MODEL PERFORMANCE ON TEST SET")
    print("="*60)
    print(f"\n🎯 Mean IoU (mIoU): {mean_iou:.4f}\n")
    print("Per-Class IoU:")
    print(iou_df.to_string(index=False, float_format="%.4f"))
    print("\n" + "="*60 + "\n")
else:
    print("⚠️  No predictions generated")

## 13. Visualize Sample Predictions

In [ ]:
# Create visualization loader
vis_loader = DataLoader(
    datamodule.test_dataset, 
    batch_size=NUM_SAMPLES_TO_TEST, 
    shuffle=True, 
    collate_fn=datamodule._collate_fn
)

print(f"\n🎨 Visualizing {NUM_SAMPLES_TO_TEST} random test samples...\n")

vis_batch = next(iter(vis_loader))

if vis_batch:
    vis_chips = vis_batch['chip'].float().to(device)
    vis_true_masks = vis_batch['mask'].float().to(device)
    vis_file_paths = vis_batch['path']

    with torch.no_grad():
        vis_logits = trained_model(vis_chips)
        vis_probs = torch.sigmoid(vis_logits.squeeze(1))
        vis_predicted_masks = (vis_probs > 0.5).float()
    
    save_dir = os.path.join(WORK_DIR, 'predictions')
    
    for i in range(len(vis_chips)):
        visualize_prediction(
            vis_chips[i], 
            vis_true_masks[i], 
            vis_predicted_masks[i], 
            vis_file_paths[i],
            save_dir
        )
    
    print(f"\n✅ All visualizations saved to: {save_dir}")
else:
    print("⚠️  No visualization batch available")

## 14. Summary

### ✅ Key Fixes Applied:

1. **Consistent Binary Task**: All components now use binary classification (2 classes: cloud/no-cloud)
2. **Fixed Visualization**: Binary colormap and labels instead of 9-class
3. **Fixed IoU Calculation**: Binary IoU for 2 classes instead of 9
4. **Fixed Class Names**: Consistent 2-class names throughout
5. **Corrected Title**: "Cloud Height Binary Segmentation" instead of "Cloud Type"
6. **Added Documentation**: Clear comments and markdown explanations

### 📈 Model Output:
- **Input**: 16-channel ABI satellite imagery
- **Output**: Binary mask (0=no cloud, 1=cloud)
- **Loss**: Combined BCE + Dice Loss
- **Metrics**: Binary Accuracy, F1-Score, IoU

### 🎯 Next Steps:
1. Monitor training progress via TensorBoard logs
2. Adjust hyperparameters if needed
3. Evaluate on different thresholds (0.3, 0.5, 0.7)
4. Consider multi-class if cloud types are important